In [2]:

import os
import re
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


# ==========================================
# 1. ファイルパス
# ==========================================

base_path = r"C:\Users\Admin\2026-\one-sentence"

human_dir = os.path.join(base_path, "Human")
gemini_dir = os.path.join(base_path, "Gemini")

train_human_path = os.path.join(
    human_dir, "train_Human_one_sentence_stylometry_features.csv"
)

test_human_path = os.path.join(
    human_dir, "test_Human_one_sentence_stylometry_features.csv"
)

train_gemini_path = os.path.join(
    gemini_dir, "train_Gemini_one_sentence_stylometry_features.csv"
)

test_gemini_path = os.path.join(
    gemini_dir, "test_Gemini_one_sentence_stylometry_features.csv"
)

output_dir = gemini_dir
os.makedirs(output_dir, exist_ok=True)


# ==========================================
# 2. 特徴量セット
# ==========================================

function_features = [
    "機能語率", "ひらがな率",
    "句点頻度", "読点頻度",
    "助詞割合", "助動詞割合",
    "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と",
    "助詞_か", "助詞_が", "助詞_で", "助詞_や",
    "助詞_を", "助詞_の", "助詞_ほど", "助詞_まで",
    "助詞_も", "助詞_から", "助詞_たり", "助詞_など",
    "助詞_ながら", "助詞_だけ", "助詞_ば", "助詞_へ",
    "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら",
    "助詞_より", "助詞_こそ", "助詞_つつ",
    "助詞_くらい", "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合",
    "形容詞割合", "副詞割合", "形状詞割合",
    "漢字率", "カタカナ率", "英数字率"
]

feature_sets = {
    "All": function_features + content_features,
    "Function": function_features,
    "Content": content_features
}


# ==========================================
# 3. 学習データの読み込み・ラベル付与
# ==========================================

df_train_human = pd.read_csv(train_human_path)
df_train_gemini = pd.read_csv(train_gemini_path)

# ラベル付与
df_train_human["label"] = 0
df_train_gemini["label"] = 1

# 学習データを結合
df_train = pd.concat(
    [df_train_human, df_train_gemini],
    ignore_index=True
)

# 文書IDと文順を作成
df_train["doc_id"] = (
    df_train["filename"]
    .str.replace(r"_s\d+$", "", regex=True)
)

df_train["sent_order"] = (
    df_train["filename"]
    .str.extract(r"_s(\d+)$")[0]
    .astype(int)
)

# HumanとGeminiの文書IDが重複しないようにする
df_train.loc[
    df_train["label"] == 0, "doc_id"
] = "Human_" + df_train.loc[
    df_train["label"] == 0, "doc_id"
].astype(str)

df_train.loc[
    df_train["label"] == 1, "doc_id"
] = "Gemini_" + df_train.loc[
    df_train["label"] == 1, "doc_id"
].astype(str)

# 文書順に並び替え
df_train = df_train.sort_values(
    ["doc_id", "sent_order"]
).reset_index(drop=True)


# ==========================================
# 4. テストデータの読み込み
# ==========================================

df_test_human = pd.read_csv(test_human_path)
df_test_gemini = pd.read_csv(test_gemini_path)


def prepare_test_data(df, prefix):

    df = df.copy()

    df["doc_id"] = (
        prefix + "_" +
        df["filename"].str.replace(
            r"_s\d+$", "", regex=True
        )
    )

    df["sent_order"] = (
        df["filename"]
        .str.extract(r"_s(\d+)$")[0]
        .astype(int)
    )

    return df.sort_values(
        ["doc_id", "sent_order"]
    ).reset_index(drop=True)


df_test_human = prepare_test_data(
    df_test_human, "Human"
)

df_test_gemini = prepare_test_data(
    df_test_gemini, "Gemini"
)

# テストデータの正解ラベル
df_test_human["label"] = 0
df_test_gemini["label"] = 1


# ==========================================
# 5. 特徴量の欠損列を補完
# ==========================================

all_features = list(dict.fromkeys(
    function_features + content_features
))

for df in [df_train, df_test_human, df_test_gemini]:

    for col in all_features:
        if col not in df.columns:
            df[col] = 0

    df[all_features] = (
        df[all_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )


# ==========================================
# 6. Logistic Regressionによる学習・確信度算出
# ==========================================

test_datasets = {
    "Human": df_test_human,
    "Gemini": df_test_gemini
}

for set_name, features in feature_sets.items():

    print(f"\n========== {set_name} ==========")

    # 学習データ
    X_train = df_train[features]
    y_train = df_train["label"]

    # 標準化
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    # Logistic Regression
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # ======================================
    # 回帰係数の出力
    # ======================================

    coef_df = pd.DataFrame({
        "特徴量": features,
        "回帰係数": model.coef_[0],
        "絶対回帰係数": np.abs(model.coef_[0])
    })

    coef_df = coef_df.sort_values(
        "絶対回帰係数",
        ascending=False
    )

    coef_df.to_csv(
        os.path.join(
            output_dir,
            f"LR_feature_coefficient_{set_name}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    # ======================================
    # Human / Gemini の確信度算出
    # ======================================

    for test_name, df_test in test_datasets.items():

        results = []

        # 文書ごとに処理
        for doc_id, group in df_test.groupby(
            "doc_id", sort=False
        ):

            group = group.sort_values("sent_order")

            proba_history = []

            for _, row in group.iterrows():

                # 各累積段階の特徴量
                X = row[features].values.reshape(1, -1)

                # 学習時と同じScalerで標準化
                X_scaled = scaler.transform(X)

                # AIクラス（label=1）の確率
                ai_proba = model.predict_proba(X_scaled)[0][
                    list(model.classes_).index(1)
                ]

                proba_history.append(ai_proba)

                # それまでのAI確率の累積平均
                cum_proba_avg = np.mean(proba_history)

                results.append({
                    "doc_id": doc_id,
                    "filename": row["filename"],
                    "sent_order": row["sent_order"],
                    "label": row["label"],
                    "AI確率": ai_proba,
                    "累積平均AI確率": cum_proba_avg,
                    "予測ラベル": int(cum_proba_avg >= 0.5)
                })

        # CSV出力
        result_df = pd.DataFrame(results)

        output_path = os.path.join(
            output_dir,
            f"Gemini_LR_{test_name}_{set_name}_results.csv"
        )

        result_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"{test_name} 出力完了: {output_path}"
        )

print("\nすべての処理が完了しました。")



========== All ==========
Human 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Human_All_results.csv
Gemini 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Gemini_All_results.csv

========== Function ==========


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.ven

Human 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Human_Function_results.csv
Gemini 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Gemini_Function_results.csv

========== Content ==========
Human 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Human_Content_results.csv


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.ven

Gemini 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_LR_Gemini_Content_results.csv

すべての処理が完了しました。


In [1]:

import os
import re
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


# ==========================================
# 1. ファイルパス
# ==========================================

base_path = r"C:\Users\Admin\2026-\one-sentence"

human_dir = os.path.join(base_path, "Human")
gemini_dir = os.path.join(base_path, "Gemini")

train_human_path = os.path.join(
    human_dir, "train_Human_one_sentence_stylometry_features.csv"
)

test_human_path = os.path.join(
    human_dir, "test_Human_one_sentence_stylometry_features.csv"
)

train_gemini_path = os.path.join(
    gemini_dir, "train_Gemini_one_sentence_stylometry_features.csv"
)

test_gemini_path = os.path.join(
    gemini_dir, "test_Gemini_one_sentence_stylometry_features.csv"
)

output_dir = gemini_dir
os.makedirs(output_dir, exist_ok=True)


# ==========================================
# 2. 特徴量セット
# ==========================================

function_features = [
    "機能語率", "ひらがな率",
    "句点頻度", "読点頻度",
    "助詞割合", "助動詞割合",
    "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と",
    "助詞_か", "助詞_が", "助詞_で", "助詞_や",
    "助詞_を", "助詞_の", "助詞_ほど", "助詞_まで",
    "助詞_も", "助詞_から", "助詞_たり", "助詞_など",
    "助詞_ながら", "助詞_だけ", "助詞_ば", "助詞_へ",
    "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら",
    "助詞_より", "助詞_こそ", "助詞_つつ",
    "助詞_くらい", "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合",
    "形容詞割合", "副詞割合", "形状詞割合",
    "漢字率", "カタカナ率", "英数字率"
]

feature_sets = {
    "All": function_features + content_features,
    "Function": function_features,
    "Content": content_features
}


# ==========================================
# 3. 学習データの読み込み・ラベル付与
# ==========================================

df_train_human = pd.read_csv(train_human_path)
df_train_gemini = pd.read_csv(train_gemini_path)

# ラベル付与
df_train_human["label"] = 0
df_train_gemini["label"] = 1

# 学習データを結合
df_train = pd.concat(
    [df_train_human, df_train_gemini],
    ignore_index=True
)

# 文書IDと文順を作成
df_train["doc_id"] = (
    df_train["filename"]
    .str.replace(r"_s\d+$", "", regex=True)
)

df_train["sent_order"] = (
    df_train["filename"]
    .str.extract(r"_s(\d+)$")[0]
    .astype(int)
)

# HumanとGeminiで文書IDが重複しないようにする
df_train.loc[
    df_train["label"] == 0, "doc_id"
] = "Human_" + df_train.loc[
    df_train["label"] == 0, "doc_id"
].astype(str)

df_train.loc[
    df_train["label"] == 1, "doc_id"
] = "Gemini_" + df_train.loc[
    df_train["label"] == 1, "doc_id"
].astype(str)

# 文書順に並び替え
df_train = df_train.sort_values(
    ["doc_id", "sent_order"]
).reset_index(drop=True)


# ==========================================
# 4. テストデータの読み込み
# ==========================================

df_test_human = pd.read_csv(test_human_path)
df_test_gemini = pd.read_csv(test_gemini_path)

# テストデータに正解ラベルを付与
df_test_human["label"] = 0
df_test_gemini["label"] = 1

# 文書IDと文順を作成する関数
def prepare_test_data(df, prefix):

    df = df.copy()

    df["doc_id"] = (
        prefix + "_" +
        df["filename"].str.replace(
            r"_s\d+$", "", regex=True
        )
    )

    df["sent_order"] = (
        df["filename"]
        .str.extract(r"_s(\d+)$")[0]
        .astype(int)
    )

    return df.sort_values(
        ["doc_id", "sent_order"]
    ).reset_index(drop=True)


df_test_human = prepare_test_data(
    df_test_human, "Human"
)

df_test_gemini = prepare_test_data(
    df_test_gemini, "Gemini"
)


# ==========================================
# 5. 特徴量の欠損列を補完
# ==========================================

all_features = list(dict.fromkeys(
    function_features + content_features
))

for df in [df_train, df_test_human, df_test_gemini]:

    for col in all_features:
        if col not in df.columns:
            df[col] = 0

    df[all_features] = (
        df[all_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )


# ==========================================
# 6. Random Forestによる学習・確信度算出
# ==========================================

test_datasets = {
    "Human": df_test_human,
    "Gemini": df_test_gemini
}

for set_name, features in feature_sets.items():

    print(f"\n========== {set_name} ==========")

    # 学習データ
    X_train = df_train[features]
    y_train = df_train["label"]

    # Random Forest
    model = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # ======================================
    # 特徴量重要度
    # ======================================

    importance_df = pd.DataFrame({
        "特徴量": features,
        "重要度": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        "重要度", ascending=False
    )

    importance_df.to_csv(
        os.path.join(
            output_dir,
            f"RF_feature_importance_{set_name}.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    # ======================================
    # Human / Gemini の確信度算出
    # ======================================

    for test_name, df_test in test_datasets.items():

        results = []

        # 文書ごとに処理
        for doc_id, group in df_test.groupby(
            "doc_id", sort=False
        ):

            group = group.sort_values("sent_order")

            proba_history = []

            for _, row in group.iterrows():

                # 各累積段階の特徴量
                X = row[features].values.reshape(1, -1)

                # AIクラス（label=1）の確率
                ai_proba = model.predict_proba(X)[0][
                    list(model.classes_).index(1)
                ]

                proba_history.append(ai_proba)

                # それまでのAI確率の累積平均
                cum_proba_avg = np.mean(proba_history)

                results.append({
                    "doc_id": doc_id,
                    "filename": row["filename"],
                    "sent_order": row["sent_order"],
                    "label": row["label"],
                    "AI確率": ai_proba,
                    "累積平均AI確率": cum_proba_avg,
                    "予測ラベル": int(cum_proba_avg >= 0.5)
                })

        # CSV出力
        result_df = pd.DataFrame(results)

        output_path = os.path.join(
            output_dir,
            f"Gemini_RF_{test_name}_{set_name}_results.csv"
        )

        result_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"{test_name} 出力完了: {output_path}"
        )

print("\nすべての処理が完了しました。")



========== All ==========


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  

Human 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_RF_Human_All_results.csv


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  

Gemini 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_RF_Gemini_All_results.csv

========== Function ==========


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  

... (4 outputs left out)

Human 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_RF_Human_Content_results.csv


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  

Gemini 出力完了: C:\Users\Admin\2026-\one-sentence\Gemini\Gemini_RF_Gemini_Content_results.csv

すべての処理が完了しました。


c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\2026-\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
